# Lab 9B: Sessions and a hosted run
**Claude Certified Developer (Foundations) · 30-Hour Program for UST · Day 9 · about 20 minutes**

**Objectives**
- Resume a session from its ID
- Fork one session into two strategies
- Measure context growth and the effect of clearing old tool results
- Write a container file and a run wrapper with limits and logging

**Before you start:** in Colab, click the key icon on the left, add a secret named `ANTHROPIC_API_KEY`, and turn on notebook access. All data in this lab is synthetic.

In [ ]:
# Setup: run once per session
!pip -q install -U anthropic 
import os, json, time, random, importlib, re
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your environment"
import anthropic
print("anthropic SDK", anthropic.__version__, "| key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

In [ ]:
%%writefile config.py
"""One place for Claude settings (same pattern as Day 1)."""
import os
# Confirm current IDs on the models overview page: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5")
FAST_MODEL = os.environ.get("CLAUDE_FAST_MODEL", "claude-haiku-4-5-20251001")
MAX_TOKENS = 1024
MAX_RETRIES = 2
TIMEOUT_S = 120

In [ ]:
import config
importlib.reload(config)
client = anthropic.Anthropic(max_retries=config.MAX_RETRIES, timeout=config.TIMEOUT_S)

def log_response(resp, label=""):
    kinds = [b.type for b in resp.content]
    print(f"{label} stop_reason={resp.stop_reason} blocks={kinds} "
          f"input_tokens={resp.usage.input_tokens} output_tokens={resp.usage.output_tokens}")
    for b in resp.content:
        if b.type == "text":
            print("   ", b.text[:300].replace("\n", " "))

def text_of(resp):
    return "".join(b.text for b in resp.content if b.type == "text")

print("model:", config.MODEL, "| fast model:", config.FAST_MODEL)

In [ ]:
# Setup: Claude Agent SDK (runs the Claude Code runtime as a library; Python 3.10+)
!pip -q install -U claude-agent-sdk
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY"
from claude_agent_sdk import (query, ClaudeSDKClient, ClaudeAgentOptions, AgentDefinition,
                              HookMatcher, tool, create_sdk_mcp_server,
                              AssistantMessage, ResultMessage, TextBlock, ToolUseBlock)
SDK_MODEL = "sonnet"   # alias or full model ID
print("Agent SDK ready")

In [ ]:
import json

async def run_sdk(prompt, options, show=True):
    """Run one Agent SDK query; print tool calls and the final result. Returns (result_msg, tool_calls)."""
    tool_calls, result = [], None
    async for msg in query(prompt=prompt, options=options):
        if isinstance(msg, AssistantMessage):
            names = []
            for b in msg.content:
                if isinstance(b, ToolUseBlock):
                    names.append(b.name); tool_calls.append((b.name, b.input))
                elif isinstance(b, TextBlock) and show:
                    print("  claude:", b.text[:300].replace("\n", " "))
            if names and show:
                print("  tool calls this turn:", names)
        elif isinstance(msg, ResultMessage):
            result = msg
    if result is not None and show:
        cost = getattr(result, "total_cost_usd", None) or 0
        print(f"[result] session={result.session_id} turns={getattr(result, 'num_turns', '?')} cost=${cost:.4f}")
        print((result.result or "")[:1500])
    return result, tool_calls

def text_result(content):
    """Shape a tool result for SDK MCP tools."""
    return {"content": [{"type": "text", "text": content if isinstance(content, str) else json.dumps(content)}]}

In [ ]:
# In-process support tools (SDK MCP server). Order records are deliberately verbose and messy.
import json
from datetime import datetime, timezone
VERIFIED, AUDIT, HOOK_LOG = set(), [], []
CUSTOMERS = {"C-1001": {"customer_id": "C-1001", "name": "Asha Menon", "email": "asha.m@example.com"},
             "C-1002": {"customer_id": "C-1002", "name": "Rahul Nair", "email": "rahul.n@example.com"}}
def _verbose(rec):
    return {**rec, **{f"internal_field_{i:02d}": f"v{i}" for i in range(35)}}      # 40+ fields, 5 matter
ORDERS = {"ORD-5001": _verbose({"order_id": "ORD-5001", "customer_id": "C-1001", "amount": 74.99, "status_code": 3, "delivered_at": 1788220800}),
          "ORD-5003": _verbose({"order_id": "ORD-5003", "customer_id": "C-1001", "amount": 89.00, "status_code": 2, "delivered_at": "20/09/2026"}),
          "ORD-5004": _verbose({"order_id": "ORD-5004", "customer_id": "C-1002", "amount": 750.00, "status_code": 3, "delivered_at": "2026-09-10T08:30:00+05:30"})}

@tool("get_customer", "Find a customer by customer ID (C-1234) or email. Returns matches and count. "
      "More than one match means ask for another identifier.", {"query": str})
async def get_customer(args):
    q = args["query"].strip().lower()
    hits = [c for c in CUSTOMERS.values() if q in (c["customer_id"].lower(), c["email"])]
    if len(hits) == 1:
        VERIFIED.add(hits[0]["customer_id"])            # the backend records a verified identity
    return text_result({"matches": hits, "count": len(hits)})

@tool("lookup_order", "Look up ONE order by ID (ORD-1234): amount (USD), status, delivery date.", {"order_id": str})
async def lookup_order(args):
    o = ORDERS.get(args["order_id"].strip().upper())
    return text_result(o if o else {"matches": [], "count": 0})

@tool("process_refund", "Submit a refund for a verified customer.",
      {"customer_id": str, "order_id": str, "amount": float, "reason": str})
async def process_refund(args):
    AUDIT.append(("REFUND EXECUTED", args))
    return text_result({"refund_id": f"RF-{len(AUDIT):04d}", "status": "submitted"})

@tool("escalate_to_human", "Hand the case to a human with a standalone summary (the human cannot see this chat).",
      {"customer_id": str, "summary": str, "amount": float, "recommended_action": str})
async def escalate_to_human(args):
    AUDIT.append(("ESCALATED", args))
    return text_result({"handoff_id": "H-0001", "queue": "billing-specialists"})

support_server = create_sdk_mcp_server("support", tools=[get_customer, lookup_order, process_refund, escalate_to_human])
SUPPORT_TOOLS = [f"mcp__support__{n}" for n in ("get_customer", "lookup_order", "process_refund", "escalate_to_human")]
print("support server ready")

## Drill 1: resume

In [ ]:
base = dict(model=SDK_MODEL, setting_sources=[], permission_mode="dontAsk", cwd="/content" if os.path.isdir("/content") else os.getcwd(),
            mcp_servers={"support": support_server}, allowed_tools=SUPPORT_TOOLS, max_turns=8)
r1, _ = await run_sdk("Look up ORD-5001 and ORD-5003 and summarize both in two lines.", ClaudeAgentOptions(**base))
SID = r1.session_id
r2, _ = await run_sdk("Which of those two orders had the larger amount?", ClaudeAgentOptions(**base, resume=SID))
print("same session continued:", r2.session_id == SID or "(some versions issue a new ID on resume)")

## Drill 2: fork into two strategies

In [ ]:
forks = {}
for name, ask_ in (("goodwill", "Propose a goodwill gesture for ORD-5001's duplicate charge, in one line."),
                   ("strict", "Propose a strictly policy-based resolution for ORD-5001, in one line.")):
    r, _ = await run_sdk(ask_, ClaudeAgentOptions(**base, resume=SID, fork_session=True), show=False)
    forks[name] = r.session_id; print(f"{name}: session {r.session_id} | {(r.result or '')[:150]}")
print("\nbase:", SID, "| forks differ from base:", all(v != SID for v in forks.values()))

## Drill 3: context growth in a long run
A hand-built loop looks up eight orders one by one. Each lookup returns a verbose record. Compare keeping every result with clearing old ones.

In [ ]:
import numpy as np
LOOKUP = [{"name": "lookup_order", "description": "Look up ONE order by ID (ORD-1234). Returns a full record.",
           "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}}]
IDS = ["ORD-5001", "ORD-5003", "ORD-5004"] * 3
def long_run(clear_old):
    messages = [{"role": "user", "content": "Look up these orders ONE AT A TIME, in order, then list each amount: " + ", ".join(IDS[:8])}]
    per_turn = []
    for _ in range(14):
        r = client.messages.create(model=config.FAST_MODEL, max_tokens=400, tools=LOOKUP, messages=messages)
        per_turn.append(r.usage.input_tokens)
        messages.append({"role": "assistant", "content": r.content})
        if r.stop_reason != "tool_use": break
        if clear_old:                       # replace older tool results with a one-line note
            for m in messages[:-1]:
                if m["role"] == "user" and isinstance(m["content"], list):
                    for blk in m["content"]:
                        if blk.get("type") == "tool_result" and not blk["content"].startswith("[cleared"):
                            rec = json.loads(blk["content"]); blk["content"] = f"[cleared] {rec.get('order_id')} amount={rec.get('amount')}"
        messages.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": b.id, "content": json.dumps(ORDERS.get(b.input["order_id"], {}))}
                                                     for b in r.content if b.type == "tool_use"]})
    return per_turn, text_of(r)
keep, ans_keep = long_run(False); clear, ans_clear = long_run(True)
print("input tokens per turn, keep all:", keep); print("input tokens per turn, cleared: ", clear)
print("total input tokens:", sum(keep), "vs", sum(clear))

Clearing keeps the facts the agent needs (IDs and amounts) while dropping 35 internal fields per record. Some APIs offer built-in context clearing for long agent runs; check the current docs.

## Drill 4: a hosting kit

In [ ]:
DOCKERFILE = """FROM python:3.12-slim
RUN useradd -m agent && pip install --no-cache-dir claude-agent-sdk
USER agent
WORKDIR /home/agent/work
# The API key is injected at runtime (for example: docker run -e ANTHROPIC_API_KEY ...), never baked into the image.
# Restrict network egress at the platform level to the hosts the agent needs.
COPY run_agent.py .
CMD ["python", "run_agent.py"]"""
open("Dockerfile", "w").write(DOCKERFILE); print(DOCKERFILE)

In [ ]:
import asyncio, uuid
COST_CEILING_USD, TIMEOUT_S = 0.50, 120
async def hosted_run(task, options):
    run_id, t0 = str(uuid.uuid4())[:8], time.perf_counter()
    try:
        result, calls = await asyncio.wait_for(run_sdk(task, options, show=False), timeout=TIMEOUT_S)
        status = "ok"
    except asyncio.TimeoutError:
        result, calls, status = None, [], "timeout"
    cost = getattr(result, "total_cost_usd", 0) or 0
    record = {"run_id": run_id, "status": status if cost <= COST_CEILING_USD else "over_budget", "turns": getattr(result, "num_turns", None),
              "tool_calls": [c[0] for c in calls], "cost_usd": round(cost, 4), "seconds": round(time.perf_counter() - t0, 1),
              "session_id": getattr(result, "session_id", None)}
    print(json.dumps(record)); return record
await hosted_run("Look up ORD-5004 and say whether it was charged twice.", ClaudeAgentOptions(**base))

## You should now have
- [ ] A resumed session
- [ ] Two forked strategies with distinct session IDs
- [ ] A context-growth comparison
- [ ] A Dockerfile and a run wrapper with limits and a structured log

## Check yourself
1. When would you start a fresh session instead of resuming?
2. Which fields would you keep when clearing old tool results?
3. What else belongs in a production run log?